# [EDA] 연속소둔로 정비 → 부품 연결과 온도 데이터 시각화

**목적** 
- 연속소둔로(`P1-CR1-CAL01`)의 정비 기록이 어떤 부품 교체로 이어졌는지 순서대로 따라가보자
- 같은 해(2024)의 온도·제어출력(TC·OP·CP)을 그래프로 확인 - 정썜 Issue PR 내용이기도 하니, 간략하게 진행하겠음
- 온도 데이터는 1년 내내 있는 것이 아니라 **14개 LOT 구간, 합계 약 99시간**만 있음
- 데이터: KAMP 공개 제조AI데이터셋을 재구성한 **교육용 재현 데이터** (실제 조업 데이터 아님)
- 프로젝트 목표 상기하세요. **고장/이상을 판정하지 않고**, 이상 후보를 찾기 위한 기초 그림과 사실만 정리하는 것임

**따라가는 순서**  
연속소둔로 정비이력  
→  
WO_NO  
→  
실제 부품교체이력  
→  
PART_CD  
→  
부품마스터(부품명·적용설비·수명·리드타임)  

**목차**
1. 데이터 불러오기
2. Step 1 · 연속소둔로 정비이력 (44건)
3. Step 2 · `WO_NO`로 부품교체이력 연결
4. Step 3 · `PART_CD`로 부품마스터 연결
5. 온도 데이터 시각화 (TC·OP·CP)
6. 온도 × 정비 × 부품을 한 장으로 보기
7. 정리

각 그래프 앞에는 **왜 이 그래프가 필요한가**, 뒤에는 **읽는 법**을 적었음

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")  # notebooks 폴더 기준 한 단계 위의 data 폴더
FIG_DIR = os.path.join("..", "figures")  # 그래프 저장 위치
os.makedirs(FIG_DIR, exist_ok=True)

# 기존 3개 (T- 센서 계측, G-02 사건 기록, G-01 기준 정보)
TEMP_CSV = os.path.join(
    DATA_DIR, "T-CR1-CAL01_온도.csv"
)  # 소둔로 온도·제어출력, 1초 주기
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")  # 강종변경 등, PLANT_CD 단위
REPAIR_CSV = os.path.join(
    DATA_DIR, "G-01_정기수리캘린더.csv"
)  # 계획 정지, PLANT_CD 단위

# 2026-09-28 추가된 4개 (상위 이슈 #43)
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")  # 설비 사전, 키 EQP_TAG
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")  # 부품 사전, 키 PART_CD
MAINT_CSV = os.path.join(
    DATA_DIR, "G-02_정비이력.csv"
)  # 정비·고장 기록, 키 WO_NO + EQP_TAG
PARTCHG_CSV = os.path.join(
    DATA_DIR, "G-02_부품교체이력.csv"
)  # 부품 교체, 키 WO_NO + PART_CD

ENC = "utf-8-sig"  # G- 로 시작하는 마스터 CSV는 BOM 있음

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 그래프 한글 깨짐 방지 - 설치된 폰트 중 있는 것을 골라 rcParams에 넣습니다
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in [
    "Malgun Gothic",
    "AppleGothic",
    "NanumGothic",
    "NanumBarunGothic",
    "Noto Sans CJK KR",
]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False  # 음수 기호가 □ 로 나오는 것 방지
rcParams["figure.dpi"] = 110

In [ ]:
# 이 노트북에서만 쓰는 추가 설정
import matplotlib.dates as mdates
from IPython.display import display

CAL_TAG = "P1-CR1-CAL01"  # 연속소둔로 설비 태그

# 정비유형별 색 - 모든 그래프에서 같은 색 = 같은 정비유형
MNT_ORDER = ["TBM", "BM", "CBM"]
MNT_COLORS = {"TBM": "#2a78d6", "BM": "#eb6834", "CBM": "#1baf7a"}

# 그래프 기본 모양: 위·오른쪽 테두리 없애고 옅은 격자
rcParams["axes.spines.top"] = False
rcParams["axes.spines.right"] = False
rcParams["axes.grid"] = True
rcParams["grid.color"] = "#e1e0d9"
rcParams["grid.linewidth"] = 0.6

# 그래프를 파일로 남기고 싶으면 각 그래프 칸의 plt.savefig(...) 줄 앞의 # 을 지우세요.

## 1. 데이터 불러오기

날짜 컬럼은 `parse_dates`로 읽어야 시간 계산(빼기, 정렬)을 할 수 있다고 함.  
아래쪽에서 이 옵션이 들어가있음.  

In [ ]:
eqp = pd.read_csv(EQP_CSV, encoding="utf-8")
part = pd.read_csv(PART_CSV, encoding="utf-8")
event = pd.read_csv(EVENT_CSV, encoding="utf-8", parse_dates=["EVT_DT"])
repair = pd.read_csv(REPAIR_CSV, encoding="utf-8", parse_dates=["STRT_DT", "END_DT"])
maint = pd.read_csv(MAINT_CSV, encoding="utf-8", parse_dates=["DETC_DT", "STRT_DT", "END_DT"])
partchg = pd.read_csv(PARTCHG_CSV, encoding="utf-8")
temp = pd.read_csv(TEMP_CSV, encoding="utf-8", parse_dates=["MEAS_DT"])

print("설비마스터   ", eqp.shape)
print("부품마스터   ", part.shape)
print("조업이벤트   ", event.shape)
print("정기수리     ", repair.shape)
print("정비이력     ", maint.shape)
print("부품교체이력 ", partchg.shape)
print("온도         ", temp.shape)

In [ ]:
# CR1 정기수리 일정 - 이후 그래프에서 회색으로 칠할 기간
cr1_repair = repair[repair["PLANT_CD"] == "CR1"]
display(cr1_repair)

## 2. Step 1 · 연속소둔로 정비이력

정비이력(maint 데이터프레임)에서 `EQP_TAG == "P1-CR1-CAL01"`만 남기고, 시간 차이 두 개를 새 컬럼으로 만들어보자

| 새 컬럼 | 계산 | 의미 |
| --- | --- | --- |
| `대기시간` | `STRT_DT - DETC_DT` | 탐지(접수) 시각부터 작업을 시작하기까지 걸린 시간.  TBM(계획정비)은 이상을 탐지한 것이 아니라 작업이 접수된 시각에 가깝다고 볼 수 있음 |
| `작업시간` | `END_DT - STRT_DT` | 작업을 시작해서 끝내기까지 걸린 시간 |

> 데이터 사전: `DETC_DT` = 이상감지시각이고 "계획정비에는 값이 없다"고 돼 있는데, 실제로는 TBM에도 값이 있음 → 확인 필요

In [ ]:
cal = maint[maint["EQP_TAG"] == CAL_TAG].copy()
cal = cal.sort_values("DETC_DT").reset_index(drop=True)

# 차이값을 시간 단위 숫자로 바꾸기
cal["대기시간"] = (cal["STRT_DT"] - cal["DETC_DT"]).dt.total_seconds() / 3600
cal["작업시간"] = (cal["END_DT"] - cal["STRT_DT"]).dt.total_seconds() / 3600



# FAIL_MODE 빈칸은 '고장이 없는 계획정비'일 수 있어서 지우지 않고 (공란)으로 채움
cal["FAIL_MODE"] = cal["FAIL_MODE"].fillna("(공란)")


# REMARK 앞뒤 공백 정리 ("  정기수리 부속작업" 처럼 앞에 공백이 붙은 값이 있음)
cal["REMARK"] = cal["REMARK"].str.strip()

print("전체 정비:", len(maint), "건 → 연속소둔로:", len(cal), "건")
display(cal.head())

### 2-1. 44건은 어떤 정비인가?

**왜 이 그래프가 필요한가?**
- 정비 44건을 한꺼번에 "정비"로 보면 안 됨
- 계획해서 한 정비(TBM), 고장 나서 한 정비(BM), 센서 징후를 보고 한 정비(CBM)는 의미가 완전히 다르다
- 우리 프로젝트가 만들려는 것은 **CBM 경보**이므로
    1. 각 유형이 몇 건인지
    2. 고장모드 / 부품교체가 얼마나 있는지
    - 크기를 알아야 뒤의 분석을 어떤 그룹으로 나눌지 정할 수 있음

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# (1) 정비유형
counts = cal["MNT_TYPE"].value_counts().reindex(MNT_ORDER)
bars = axes[0].bar(counts.index, counts.values, color=[MNT_COLORS[t] for t in MNT_ORDER])
axes[0].bar_label(bars)  # 막대 위에 건수 표시
axes[0].set_title("정비유형 (MNT_TYPE)")

# (2) 고장모드
counts = cal["FAIL_MODE"].value_counts()
bars = axes[1].bar(counts.index, counts.values, color="#8c8c8c")
axes[1].bar_label(bars)
axes[1].set_title("고장모드 (FAIL_MODE)")

# (3) 부품교체 여부
counts = cal["PART_CHG"].value_counts()
bars = axes[2].bar(counts.index, counts.values, color="#8c8c8c")
axes[2].bar_label(bars)
axes[2].set_title("부품교체 여부 (PART_CHG)")

for ax in axes:
    ax.set_ylabel("건수")
fig.suptitle("연속소둔로 정비 44건 구성", y=1.04)
plt.savefig(os.path.join(FIG_DIR, "03_mnt_counts.png"), bbox_inches="tight")
plt.show()

**왜 교차표도 보는가?**
- "고장모드가 비어 있는 정비는 어떤 정비유형인가?"처럼 
- **두 항목의 관계**는 교차표로 봐야함

In [ ]:
# 정비유형 × 고장모드 교차표 (칸 안의 숫자 = 건수)
cross = pd.crosstab(cal["MNT_TYPE"], cal["FAIL_MODE"]).reindex(MNT_ORDER)
display(cross)

**읽는 법:**
- TBM 22건, BM 12건, CBM 10건, 부품교체는 8건
-`FAIL_MODE` 공란 24건 중 20건이 TBM(계획정비)
- 고장이 나서 한 정비가 아니므로 고장모드가 **구조적으로 비어 있는 것** (데이터 사전에도 "FAIL_MODE 결측은 구조적"이라고 나옴)
- 결측이라고 지우면 TBM이 거의 다 사라지므로 `(공란)`으로 추가하자
- 단, BM / CBM의 공란 4건은 REMARK가 "원인 미상", "확인 중"처럼 **원인이 아직 정해지지 않은 것**이라 성격이 다르다는 거 인지하셈
- `FAIL_MODE` 코드 (데이터 사전): `ELE` 전기 / `WER` 마모 / `MAL` 정렬불량 / `ETC` 기타
    - 단, 고장모드 배정 자체가 설정값(`[가정]`)이라 실제 고장 통계는 아님

### 2-2. 탐지 → 착수 대기시간은 왜 천차만별인가?

**왜 이 그래프가 필요한가?**
- CSV를 눈으로 보면 이상탐지(`DETC_DT`)와 작업착수(`STRT_DT`) 사이가 몇 분인 경우도, 며칠인 경우도 있다
- 이 차이가 **무엇 때문인지**(`MNT_TYPE`? `FAIL_MODE`? `PART_CHG`?) 알아야, "센서 징후가 정비보다 얼마나 먼저 잡히는가"를 제대로 해석할 수 있다
- 세 기준을 **같은 y축에 나란히** 놓으면 어느 기준이 대기시간을 가장 잘 가르는지 한눈에 비교 가능

그래프 보는 법
- **점 하나 = 정비 1건**
- claude 피셜: 점 색은 세 그래프 모두 정비유형
    - 각 정비이력의 대기시간 차이가 크다
    - 그래서 y축은 로그 눈금(1 → 10 → 100시간)을 사용했다고 함

In [ ]:
np.random.seed(0)  # 점을 흩뿌리는 위치가 실행할 때마다 같도록 고정

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)


for ax, col in zip(axes, ["MNT_TYPE", "FAIL_MODE", "PART_CHG"]):
    if col == "MNT_TYPE":
        groups = MNT_ORDER
    else:
        groups = sorted(cal[col].unique())

    # 상자그림: 가운데 선이 중앙값
    data = [cal.loc[cal[col] == g, "대기시간"] for g in groups]
    ax.boxplot(data, tick_labels=groups, showfliers=False, widths=0.5)

    # 점: 정비 1건, 색 = 정비유형 (겹치지 않게 옆으로 살짝 흩뿌림)
    for i, g in enumerate(groups, start=1):
        sub = cal[cal[col] == g]
        x = i + np.random.uniform(-0.15, 0.15, len(sub))
        ax.scatter(x, sub["대기시간"], c=sub["MNT_TYPE"].map(MNT_COLORS), s=35, edgecolor="white", zorder=3)

    ax.set_yscale("log")
    ax.set_yticks([0.01, 0.1, 1, 10, 100], ["0.01", "0.1", "1", "10", "100"])  # 눈금을 보통 숫자로
    ax.set_title(f"{col}별 대기시간")



axes[0].set_ylabel("이상탐지 → 작업착수 (시간, 로그 눈금)")


for t in MNT_ORDER:  # 범례만 만들기 위해 데이터 없는 빈 점을 찍음
    axes[2].scatter([], [], c=MNT_COLORS[t], label=t)
axes[2].legend(title="정비유형", loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.savefig(os.path.join(FIG_DIR, "03_wait_hours_by_group.png"), bbox_inches="tight")
plt.show()

In [ ]:
# 숫자로도 확인: 건수 / 중앙값 / 최댓값 (단위: 시간)
display(cal.groupby("MNT_TYPE")["대기시간"].agg(["count", "median", "max"]).reindex(MNT_ORDER).round(2))
display(cal.groupby("PART_CHG")["대기시간"].agg(["count", "median", "max"]).round(2))

**읽는 법**
- 대기시간을 가장 잘 가르는 기준은 **`MNT_TYPE`**
- TBM과 BM은 모두 수 시간 안에 착수하지만
### **CBM은 중앙값이 **약 117시간(약 5일)**, 최대 약 216시간**

- `FAIL_MODE`와 `PART_CHG` 그래프에서 위쪽에 떠 있는 점들은 **CBM(초록색)** 임
    - 이 두 기준에서 보이는 차이도 대부분 `CBM`이 어느 그룹에 섞였느냐에 따라 생김. 오히려 부품교체 Y가 N보다 짧다.
- 사실만 정리하면 **BM·TBM은 모두 수 시간 안에 착수하고, CBM만 며칠 뒤에 착수**했다고 볼 수 있다.

### 해석(가설)
- CBM은 징후를 보고 작업을 **계획해서** 나중에 착수하기 때문일수도
    - 그렇다면 CBM의 대기시간은 "센서 징후가 실제 작업보다 먼저 잡힌 시간"이고, 우리 이상탐지가 만들어야 할 "미리 알려주는 시간"과 같은 개념이 된다!
- 다만 반대 근거도 있음
    - CBM 중 REMARK가 **"설비 트립"인 2건도 약 68시간·110시간 뒤에 착수**  - 설비 트립(trip)은 설비가 보호장치에 걸려 자동으로 멈춘 것
    - "확인 중" OR "원인 미상" OR "재점검 필요" -> 원인을 찾는 중이었던 기록이 많다. 
        - 반대로 CBM 10건 중 2건은 하루 안에 착수했음 -> 그래서 "계획된 여유"라고 단정하지 말아야 함.

### 보고서에는 그래서 보고서에는 "계획된 여유일 수도 있다(가설)" 정도로만 쓰자는 의미가 될 수 있다 ㅇㅇ

### 2-3. 1년 동안 언제 정비했나?

**왜 이 그래프가 필요한가?**
- 정비 44건이 1년에 고르게 퍼져 있는지, **특정 시기에 몰려 있는지** 봐야함 
- 특히 CR1 정기수리(6월, 11월) 기간에 한 정비는 이상 징후 때문이 아니라 수리 일정 때문에 한 것이라, 뒤에서 온도 이상 후보와 비교할 때 **섞이면 안 되는 정비**
- 이 그래프로 그런 정비를 눈으로 먼저 걸러냄

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.2))

# 회색 = CR1 정기수리 (종료일 당일까지 포함하려고 +1일)
for _, r in cr1_repair.iterrows():
    ax.axvspan(r["STRT_DT"], r["END_DT"] + pd.Timedelta(days=1), color="#d9d9d9", lw=0)

# 정비유형별로 한 줄씩 점 찍기
for i, t in enumerate(MNT_ORDER):
    sub = cal[cal["MNT_TYPE"] == t]
    ax.scatter(sub["DETC_DT"], [i] * len(sub), color=MNT_COLORS[t], s=45, edgecolor="white", zorder=3, label=t)

ax.set_yticks([0, 1, 2], MNT_ORDER)
ax.set_ylim(-0.7, 2.7)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
ax.set_title("연속소둔로 정비 탐지 시점 (회색 = CR1 정기수리)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.savefig(os.path.join(FIG_DIR, "03_mnt_timeline.png"), bbox_inches="tight")
plt.show()

In [ ]:
# 정기수리 기간(종료일 포함) 안에 탐지된 정비
cal["정기수리중"] = False
for _, r in cr1_repair.iterrows():
    inside = (cal["DETC_DT"] >= r["STRT_DT"]) & (cal["DETC_DT"] < r["END_DT"] + pd.Timedelta(days=1))
    cal.loc[inside, "정기수리중"] = True

print("정기수리 기간 안에 탐지된 정비:", cal["정기수리중"].sum(), "건")
display(cal.loc[cal["정기수리중"], ["WO_NO", "DETC_DT", "MNT_TYPE", "FAIL_MODE", "REMARK"]])

**읽는 법:** 
- 회색(정기수리) 기간에 탐지된 11건은 모두 **TBM(파란색)**
- 그중 9건은 REMARK가 "수리기간 중 점검", "정기수리 부속작업", "SD작업 수행"처럼 **수리 일정에 따른 작업**
- 나머지 2건은 실제 결함 기록이 있음(`WO-2024-00931` 마모량 초과, `WO-2024-00916` O링 세트 손상 교체)
- 이상 징후 때문에 한 정비와 섞어서 해석하지 않도록 `정기수리중` 컬럼으로 표시해 두고, 6-2에서 이 컬럼을 사용

> 참고: 데이터 사전상 `END_DT`는 **당일 포함** → 종료일까지 넣은 11건이 맞음 (종료일 00:00에서 끊으면 9건으로 잘못 셈)

## 3. Step 2 · `WO_NO`로 부품교체이력 연결

- 부품교체이력에는 **날짜가 없음** 
- `WO_NO`(작업지시번호)로 정비이력에 붙여야 언제, 어떤 정비에서 교체했는지 알 수 있음

**왜 먼저 두 파일이 맞는지 확인하는가?** 
- 연결 키가 한쪽에만 있거나 `PART_CHG` 표시와 실제 교체기록이 어긋나면, 붙인 결과를 믿을 수 없음.
    - 연결 전에 전체 데이터에서 한 번 확인해야 함

In [ ]:
wo_y = set(maint.loc[maint["PART_CHG"] == "Y", "WO_NO"])  # 정비이력: 부품교체 = Y 인 작업지시
wo_chg = set(partchg["WO_NO"])                            # 부품교체이력에 등장한 작업지시

print("정비이력 PART_CHG=Y:", len(wo_y), "건 / 부품교체이력 WO_NO:", len(wo_chg), "개")
print("Y인데 교체기록 없음:", len(wo_y - wo_chg), "건")
print("교체기록 있는데 Y가 아님:", len(wo_chg - wo_y), "건")
print("작업지시 1건당 교체 부품 행 수 최대:", partchg["WO_NO"].value_counts().max())

In [ ]:
cal_chg = cal.merge(partchg, on="WO_NO", how="inner")

print("연속소둔로 PART_CHG=Y:", (cal["PART_CHG"] == "Y").sum(), "건")
print("부품교체가 붙은 정비:", cal_chg["WO_NO"].nunique(), "건 → 교체 행:", len(cal_chg), "행")
display(cal_chg[["WO_NO", "EQP_TAG", "DETC_DT", "MNT_TYPE", "FAIL_MODE", "PART_CD", "QTY", "CHG_RSN", "REMARK"]])

**왜 정비유형 × 교체사유를 보는가?**
- 같은 부품 교체라도 "고장 나서" 바꾼 것과 "수명이 돼서/예방으로" 바꾼 것은 의미가 다르다.
- 손상 교체가 어떤 정비유형에서 나왔는지 알면, 센서로 **미리 잡았어야 할 교체**가 무엇인지 좁힐 수 있다.

In [ ]:
cross = pd.crosstab(cal_chg["MNT_TYPE"], cal_chg["CHG_RSN"]).reindex(MNT_ORDER)
display(cross)

**읽는 법** 
- 정비 8건에서 9행이 나온 이유는 `WO-2024-00916` 한 건에서 부품 2종을 교체했기 때문
- 손상 교체 4건 중 3건이 BM(고장정비)
- `QTY` = **한 작업지시에서 해당 부품을 교체한 수량(개)** (데이터 사전 정의)

## 4. Step 3 · `PART_CD`로 부품마스터 연결

부품 코드에 이름 | 분류 |적용설비 | 표준수명(`STD_LIFE_H`, 시간) | 조달기간(`LEAD_TIME_D`, 일)을 붙여보자

In [ ]:
cal_part = cal_chg.merge(part, on="PART_CD", how="left")
print("부품마스터에서 못 찾은 PART_CD:", cal_part["PART_NM"].isna().sum(), "건")

show_cols = ["WO_NO", "DETC_DT", "MNT_TYPE", "PART_CD", "PART_NM", "PART_TYPE",
             "APPLY_EQP_TYPE", "STD_LIFE_H", "LEAD_TIME_D", "QTY", "CHG_RSN"]
display(cal_part[show_cols])

In [ ]:
# 부품별 요약: 교체 횟수(count)와 총 수량(sum)
display(cal_part.groupby(["PART_CD", "PART_NM"])["QTY"].agg(["count", "sum"]))

### 4-1. 어떤 부품을 언제 교체했나?

**왜 이 그래프가 필요한가?**
- 표로는 9행이 흩어져 보이지만, 부품별로 한 줄씩 날짜 위에 찍으면 **같은 부품이 몇 번, 얼마 간격으로 반복 교체되는지**가 바로 보임
- 또 교체가 정기수리 기간에 몰렸는지, 고장정비(BM)에서 나왔는지도 함께 확인할 수 있음

그래프 보는 법: 점 색 = 정비유형(TBM, BM, CBM), 옆 글자 = `교체사유 ×수량`.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

for _, r in cr1_repair.iterrows():
    ax.axvspan(r["STRT_DT"], r["END_DT"] + pd.Timedelta(days=1), color="#d9d9d9", lw=0)

for t in MNT_ORDER:
    sub = cal_part[cal_part["MNT_TYPE"] == t]
    ax.scatter(sub["END_DT"], sub["PART_NM"], color=MNT_COLORS[t], s=90, edgecolor="white", zorder=3, label=t)

for _, r in cal_part.iterrows():
    ax.annotate(f"{r['CHG_RSN']} ×{r['QTY']}", (r["END_DT"], r["PART_NM"]),
                xytext=(6, 6), textcoords="offset points", fontsize=8)

ax.set_xlim(pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-25"))  # 12월 글자가 잘리지 않게 오른쪽 여유
ax.set_ylim(-0.6, 2.6)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
ax.set_title("연속소둔로 부품 교체 시점 (작업완료 END_DT 기준, 회색 = CR1 정기수리)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.savefig(os.path.join(FIG_DIR, "03_part_change_timeline.png"), bbox_inches="tight")
plt.show()

**읽는 법**
- 교체된 부품은 **3종뿐**이고, 3종 모두 1년에 **3번씩** 교체되었음
- 6월 10일(정기수리 첫날)에는 오일씰과 O링 세트를 같이 바꿨음

### 4-2. 표준수명과 비교하면?

**왜 이 그래프가 필요한가?**
- 부품마스터의 표준수명(`STD_LIFE_H`)은 "보통 이만큼 쓴다"는 기준
- 실제로 다시 교체하기까지 걸린 시간이 표준수명보다 **훨씬 짧다면**, 그 부품은 예상보다 빨리 닳거나 자주 손상되는 **조기 교체 후보**일 수 있음
- 이상탐지로 미리 잡을 가치가 있는 부품을 고르는 근거가 됩니다. 단, 교체 사유(손상/수명도래/예방)도 함께 봐야 함

계산: 같은 부품의 `직전 교체 완료(END_DT) → 이번 교체 완료`   
- 부품마다 첫 교체는 그 전 설치일을 몰라서 계산에서 뺴겠음

In [ ]:
cal_part = cal_part.sort_values("END_DT")
cal_part["직전교체후_h"] = cal_part.groupby("PART_CD")["END_DT"].diff().dt.total_seconds() / 3600
cal_part["수명대비_%"] = cal_part["직전교체후_h"] / cal_part["STD_LIFE_H"] * 100

gaps = cal_part.dropna(subset=["직전교체후_h"])
display(gaps[["PART_CD", "PART_NM", "END_DT", "CHG_RSN", "직전교체후_h", "STD_LIFE_H", "수명대비_%"]].round({"직전교체후_h": 1, "수명대비_%": 1}))

In [ ]:
life = cal_part.groupby("PART_NM")["STD_LIFE_H"].first()  # 부품별 표준수명

fig, ax = plt.subplots(figsize=(11, 3))
ax.barh(life.index, life.values, color="#d9d9d9", height=0.5, label="표준수명 (STD_LIFE_H)")
ax.scatter(gaps["직전교체후_h"], gaps["PART_NM"], color="#2a78d6", s=70, zorder=3, label="실제 재교체 간격")
ax.set_xlabel("시간 (h)")
ax.set_title("표준수명(회색 막대) vs 실제 재교체 간격(파란 점)")
ax.legend(loc="lower right")
plt.savefig(os.path.join(FIG_DIR, "03_life_vs_interval.png"), bbox_inches="tight")
plt.show()

**읽는 법** 
- 세 부품 모두 표준수명보다 **훨씬 짧은 간격(표준수명의 40% 미만)** 으로 다시 교체되었음 (가정: 표준수명이 가동시간 기준이라면, 여기서 잰 달력 시간보다 실제 가동시간은 짧으므로 간격은 더 짧아진다. 가동시간인지 달력시간인지는 확인 필요)

그래도 "빨리 닳았다"고 단정할 수 없는 이유가 있다
- 데이터 사전: `STD_LIFE_H`는 설계 표준값(설정값)이라 실제 교체 주기와 원래 일치하지 않음
- 재교체 6번 중 **손상 교체는 3번뿐**이고, 나머지는 예방 2번, 수명도래 1번
    - 예방 교체는 닳아서 바꾼 것이 아님
- 표준수명의 23%밖에 안 됐는데 사유가 "수명도래"인 기록(오일씰, 6/10)이 있어, `CHG_RSN`과 `STD_LIFE_H`의 수명 기준이 서로 다를 수 있음
- `QTY`가 1~3개로 다르고 **어느 위치의 부품인지** 기록이 없습니다.

### 그래서 "조기 교체가 반복되는 부품 후보"로만 두자

### 4-3. 연속로 전용 부품 중 실제로 교체된 것은?

**왜 이 그래프가 필요한가?**
- 부품마스터에는 "공통" 57개, "연속로" 11개가 있다 
- 이 68개를 전부 분석할 필요가 있는지, 그리고 **온도 채널과 직접 관련된 부품(히터, 열전대 등)** 이 실제로 교체된 적이 있는지 확인해야 함 
- 온도 이상 후보를 나중에 "부품 교체 기록"으로 검증할 수 있는지가 여기서 결정됨

그래프 보는 법
- 연속로 전용 부품 11개
    - 파랑 = 2024년 CAL01에서 교체됨
    - 회색 = 교체 기록 없음.

In [ ]:
furnace = part[part["APPLY_EQP_TYPE"] == "연속로"].copy()
furnace["교체됨"] = furnace["PART_CD"].isin(cal_part["PART_CD"])
furnace = furnace.sort_values("STD_LIFE_H")

labels = furnace["PART_CD"] + " " + furnace["PART_NM"]
colors = ["#2a78d6" if x else "#c3c2b7" for x in furnace["교체됨"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
axes[0].barh(labels, furnace["STD_LIFE_H"], color=colors)
axes[0].set_title("표준수명 STD_LIFE_H (시간)")
axes[1].barh(labels, furnace["LEAD_TIME_D"], color=colors)
axes[1].set_title("조달기간 LEAD_TIME_D (일)")
fig.suptitle("연속로 전용 부품 11개 (파랑 = 2024년 CAL01에서 교체, 회색 = 교체 기록 없음)", y=1.02)
plt.savefig(os.path.join(FIG_DIR, "03_furnace_parts.png"), bbox_inches="tight")
plt.show()

print("공통 + 연속로 부품 수:", part["APPLY_EQP_TYPE"].isin(["공통", "연속로"]).sum(), "개")
print("그중 CAL01에서 실제 교체된 부품:", sorted(cal_part["PART_CD"].unique()))

**읽는 법 (도메인 관점)**

- 실제 교체된 부품은 **오일씰(SEL-006, 공통), O링 세트(SEL-009, 공통), 가스 분석계 셀(INS-006, 연속로)** 3종뿐
    - 공통/연속로 후보 68개 중 **교체 기록으로 검증할 수 있는 것은 이 3종뿐**
    - 오일씰과 O링은 공장 전체에서 29번, 36번 교체된 흔한 공통 실링 부품
    -  연속소둔로에서 이 부품을 바꾼 BM은 고장모드가 ELE(인버터 알람·모터 과열·절연 저하) 
        - 온도 채널(TC·OP·CP)과 직접 이어지는 근거는 없다
---
- 온도 채널과 가장 직접 관련된 **전기히터 소자 Z1~Z3, 열전대, CP 산소 프로브, 라디언트 튜브**는 2024년 교체 기록이 없다
    - 그래서 TC·OP 채널에서 이상 후보가 나와도 "히터/열전대를 교체했다"는 기록으로 검증할 사례는 없다
    - 하지만! 교체 기록이 없다고 **열화가 없었다는 뜻은 아니며**, 교체 없이 서서히 나빠지는 것을 찾는 것이 바로 이상탐지의 역할이 된다

---

- `CP-Z4`는 데이터 사전상 **카본포텐셜** (가열·균열존 분위기 제어값). 단위와 구체적 의미는 확인 필요 (`[확인필요]`) 
    - 가스 분석계 셀(INS-006)도 분위기 계측 부품이라 `CP-Z4`와 연결해 볼 후보 (사전에 둘의 관계는 없음 → 추정)

- 교체된 3종 중 조달기간이 가장 긴 가스 분석계 셀(31일)은, 예비품이 없다면 경보가 **조달기간보다 먼저** 나와야 의미가 있다
    - 연속로 부품 중에는 퍼니스롤(85일), 라디언트 튜브(72일)처럼 더 긴 부품도 있습니다. CBM 경보 기준을 정할 때 쓸 수 있는 값임

## 5. 온도 데이터 시각화

### 5-1. LOT 요약과 강종 붙이기

LOT_NO가 변경될 때의 시작 시각은 조업이벤트.csv의 `CR1 강종변경` 이벤트 `EVT_DT`와 **분 단위까지** 같다 - 이건 뭐 여러분들도 아는 내용이니까 ㅇㅇ
- 조업이벤트.csv는 분까지만 기록되어 있고, LOT 시작은 모두 00초
- 그래서 시작 시각이 같은 이벤트를 찾아 **변경 후 강종(`AFT_VAL`)** 을 LOT에 붙여서 시각화 해볼거임
- 강종에 따라 `정상 온도`와 `정상 출력` 수준이 다르므로 => 이후 모든 그래프에 강종을 같이 적어 **강종 차이를 이상으로 오해하지 않도록** 해보자.

In [ ]:
# LOT별 시작·끝·행 수
lot = temp.groupby("LOT_NO")["MEAS_DT"].agg(["min", "max", "count"]).reset_index()
lot.columns = ["LOT_NO", "시작", "끝", "행수"]
lot["길이_h"] = ((lot["끝"] - lot["시작"]).dt.total_seconds() / 3600).round(2)

# 시작 시각 = CR1 강종변경 시각 → 변경 후 강종(AFT_VAL)을 붙입니다
cr1_grade = event[(event["PLANT_CD"] == "CR1") & (event["EVT_TYPE"] == "강종변경")]
lot = lot.merge(cr1_grade[["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL"]], left_on="시작", right_on="EVT_DT", how="left")
lot = lot.rename(columns={"BEF_VAL": "이전_강종", "AFT_VAL": "강종"}).drop(columns="EVT_DT")
lot = lot.sort_values("시작").reset_index(drop=True)

print("강종을 못 붙인 LOT:", lot["강종"].isna().sum(), "개")
display(lot)

# 온도 데이터에도 강종 컬럼 추가
grade_of = lot.set_index("LOT_NO")["강종"]  # LOT 번호 → 강종 (grade_of[240061] 처럼 씀)
temp["강종"] = temp["LOT_NO"].map(grade_of)

# 그래프 x축 이름표로 쓸 "LOT 번호 + 줄바꿈 + 강종" (LOT 시간 순서)
xlabels = (lot["LOT_NO"].astype(str) + "\n" + lot["강종"]).tolist()

#### LOT_NO ↔ 강종(AFT_VAL) 대응이 맞는지 확인

**왜 확인이 필요한가?**
- "LOT 시작 시각 = CR1 강종변경 시각"이라는 한 가지 근거로 `AFT_VAL`을 강종으로 붙였는데,
    - 이 노트북에선 두 가지를 더 확인해보겠음

1. **LOT 도중에 강종이 또 바뀌지 않았나?** 
    - LOT 관측 구간 안의 강종변경이 시작 시각의 1건뿐이어야 LOT 하나 = 강종 하나
2. **LOT가 끝날 때까지 그 강종이 유지됐나?** 
    - LOT가 끝난 뒤 처음 나오는 강종변경의 "변경 전 강종(`BEF_VAL`)"이 이 LOT의 강종과 같으면, LOT 동안 강종이 그대로였다는 뜻

In [ ]:
check = lot[["LOT_NO", "시작", "끝", "EVT_ID", "이전_강종", "강종"]].copy()
check["구간안_강종변경"] = 0
check["다음변경의_BEF_VAL"] = ""

for i, r in check.iterrows():
    # 1) LOT 관측 구간 안에 있는 강종변경 건수
    inside = (cr1_grade["EVT_DT"] >= r["시작"]) & (cr1_grade["EVT_DT"] <= r["끝"])
    check.loc[i, "구간안_강종변경"] = inside.sum()

    # 2) LOT가 끝난 뒤 처음 나오는 강종변경의 '변경 전 강종'
    later = cr1_grade[cr1_grade["EVT_DT"] > r["끝"]].sort_values("EVT_DT")
    check.loc[i, "다음변경의_BEF_VAL"] = later["BEF_VAL"].iloc[0]

check["강종_유지"] = check["강종"] == check["다음변경의_BEF_VAL"]
display(check)

print("구간 안 강종변경이 1건이 아닌 LOT:", (check["구간안_강종변경"] != 1).sum(), "개")
print("LOT 뒤 다음 변경의 BEF_VAL이 강종과 다른 LOT:", (~check["강종_유지"]).sum(), "개")
print("강종이 비어 있는 온도 행:", temp["강종"].isna().sum(), "행 /", len(temp), "행")

**강종별로 묶어 보기**

**왜 이 그래프가 필요한가?** 
- LOT → 강종은 하나로 정해지지만, 강종 → LOT는 여러 개일 수 있다는 의심을 해볼 수 있으니까 ㅇㅇ
- 강종마다 LOT가 몇 개, 몇 시간 관측됐는지 알아야 "같은 강종끼리 비교"(5-4, 6-4)를 할 수 있는 강종이 어느 것인지 `정확하게 정의`할 수 있음
- LOT가 1개뿐인 강종은 비교할 상대가 없다

In [ ]:
grade_summary = lot.groupby("강종").agg(LOT수=("LOT_NO", "count"), 관측시간_h=("길이_h", "sum"))
grade_summary = grade_summary.sort_values("관측시간_h")

# 강종별 LOT 번호 목록
for g in grade_summary.index:
    print(f"{g:8s} → {lot.loc[lot['강종'] == g, 'LOT_NO'].tolist()}")

# 이벤트에는 나오지만 온도가 관측된 LOT는 없는 강종
event_grades = set(cr1_grade["BEF_VAL"]) | set(cr1_grade["AFT_VAL"])
print("이벤트에만 있고 온도 LOT는 없는 강종:", sorted(event_grades - set(lot["강종"])))

fig, ax = plt.subplots(figsize=(10, 3.5))
bars = ax.barh(grade_summary.index, grade_summary["관측시간_h"], color="#2a78d6")
ax.bar_label(bars, labels=(grade_summary["LOT수"].astype(str) + "개 LOT").tolist(), padding=4, fontsize=9)
ax.set_xlabel("관측 시간 합계 (h)")
ax.set_title("강종별 온도 관측 시간과 LOT 수")
plt.savefig(os.path.join(FIG_DIR, "03_grade_lot_summary.png"), bbox_inches="tight")
plt.show()

**읽는 법**

- 14개 LOT 모두 관측 구간 안의 강종변경은 **시작 시각의 1건뿐**
- LOT가 끝난 뒤 다음 강종변경의 `BEF_VAL`도 **14개 모두 같은 강종**
- 앞뒤 양쪽에서 확인되므로 **LOT 1개 = 강종 1개**로 봐도 된다 / 온도 345,390행 모두 강종이 붙어있다
- 반대로 **강종 1개 = LOT 여러 개**
    - SPHC 4개 [ 240044 / 240061 / 240069 / 240091 ]
    - SPCE 3개[ 240037 / 240121 / 240133 ]
    - SPCEN 2개
    - SPFH590 2개
    - SGCC(240001) DP590(240058) SPCD(240117)는 1개

### 디코로 말했던대로 `SPCC`는 CR1 이벤트에는 나오지만 온도가 관측된 LOT는 없슴
    - SPCC 생산이 없었다는 뜻이 아니라, 그 기간의 온도가 이 파일에 없다는 뜻
- 주의
    - 시각 일치는 **분 단위** 
        - 강종변경이 `실제 투입 시각`인지 `계획 시각`인지는 바로 아래 참고 항목처럼 아직 확인이 필요

- 정썜 노트북 `02_eda_7_steel_grade_temperature_Jinny`에도 같은 방식의 LOT–강종 연결이 있다
    - 결과는 14 LOT | 7종 | SPCC 미관측 | SGCC·DP590·SPCD 각 1 LOT => 같음

**참고로 확인할 것:**
- 정기수리 기간(설비가 멈춘 기간)에도 
- CR1 강종변경 이벤트가 기록되어 있는지 봐보자
- 있다면 이벤트가 "실제 소재 투입"이 아니라 "계획·지시" 시각일 수 있다는 신호

In [ ]:
# 정기수리 기간(종료일 포함) 안에 기록된 CR1 강종변경
for _, r in cr1_repair.iterrows():
    inside = (cr1_grade["EVT_DT"] >= r["STRT_DT"]) & (cr1_grade["EVT_DT"] < r["END_DT"] + pd.Timedelta(days=1))
    print(r["PLAN_ID"], r["SCOPE"], "→", inside.sum(), "건")
    display(cr1_grade.loc[inside, ["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL"]])

**읽는 법**
- 두 정기수리 기간 모두 강종변경이 2건씩 기록돼 있음 (그 기간 온도 데이터는 0행)
- 설비가 멈춘 동안 강종이 바뀌었다는 건 앞뒤가 안 맞음
    - 강종변경 이벤트가 `실제 투입 시각`인지 `계획 시각`인지는 데이터 제공자에게 확인할 항목으로 남기자
    - 데이터 사전에도 이벤트 시각·내용은 "설정값"이라고만 돼 있고 업무 의미 정의는 없음

### 5-2. 채널 목록과 값의 범위

채널 이름 앞글자로 나눔 (데이터 사전 기준)
- `TC` = 존별 온도(℃), `OP` = 존별 제어출력(%), `CP-Z4` = 카본포텐셜(단위 확인 필요), `CLN-IN` = 세정조 입측 온도
- 존 구분: Z1~Z2 건조존 / Z3~Z6 가열·균열존 / Z7~Z9 과시효대 / OUT1~2 냉각대 (존 배정은 사전에서도 `[가정]`)

**왜 그래프 전에 표부터 보는가?**
- 채널마다 단위와 크기가 완전히 다름 (예: 100℃ vs 860℃ vs 0.45) → 한 그래프에 같이 못 그림
- 범위·결측부터 보고 **어떤 채널끼리 묶을지**, **뺄 채널은 없는지** 정하자

In [ ]:
SENSOR_COLS = [c for c in temp.columns if c not in ["MEAS_DT", "LOT_NO", "강종"]]
TC_COLS = [c for c in SENSOR_COLS if c.startswith("TC-")]
OP_COLS = [c for c in SENSOR_COLS if c.startswith("OP-")]
CP_COLS = [c for c in SENSOR_COLS if c.startswith("CP-")]
print("TC", len(TC_COLS), TC_COLS)
print("OP", len(OP_COLS), OP_COLS)
print("CP", len(CP_COLS), CP_COLS)

stats = temp[SENSOR_COLS].describe().T
stats["결측"] = temp[SENSOR_COLS].isna().sum()
display(stats.round(3))

In [ ]:
# 눈에 띄는 값 몇 가지를 직접 세어 보자
print("CP-Z4M 최댓값:", temp["CP-Z4M"].max(), "→ 값 자체가 아주 작음")
print("CP-Z4M과 TC-Z5의 상관계수:", round(temp["CP-Z4M"].corr(temp["TC-Z5"]), 6))  # 1에 가까우면 같이 움직임
print("OP-Z3 = 100% (출력 포화) 행 수:", (temp["OP-Z3"] >= 100).sum())

# OP 8개가 동시에 0인 행
all_op_zero = (temp[OP_COLS] == 0).all(axis=1)
display(temp.loc[all_op_zero, ["MEAS_DT", "LOT_NO", "강종"] + OP_COLS])

**읽는 법**

- `CP-Z4M`은 값이 1e-10 수준으로 아주 작고, **`TC-Z5`와 상관계수가 거의 1**
    - 즉 새로운 정보가 없는 `TC-Z5`의 복사본 같은 채널이라 분석 대상에서 뺌(이름과 달리 `CP-Z4`와는 상관이 약 0.33뿐)

- OP 채널 중 `OP-Z3`만 100%(출력 최대)에 붙는 행이 많음
    -히터가 더 올릴 수 없는 MAX값이라 5-4에서 따로 봐보자
- OP 8개가 **동시에 0**이 되는 행이 2개 있고, 각각 6월·11월 정기수리 **바로 전 LOT의 마지막 시각**
    - 수리를 앞두고 가열을 끈 순간으로 보임
- 온도 수준은 Z1~Z2 약 100℃, Z3~Z6 약 860℃, Z7~Z9 약 330℃, 출구 약 280℃로 크게 다름
    - 그래서 채널마다 **자기 y축**으로 그려보자

### 5-3. 채널별 1년 전체 흐름 (14개 LOT 이어 붙이기)

**왜 이 그래프가 필요한가?**
- "각 TC·OP 채널에 이상한 값이 있는가?"를 보려면 먼저 **모든 채널의 전체 모습을 한 번에** 봐야함
    - 채널을 위아래로 쌓아 두면 잘보이지 ㅇㅇ. 정쌤 그래프처럼

In [ ]:
# LOT마다 1분 평균을 내서 다시 합칩니다
pieces = []
for lot_no in lot["LOT_NO"]:
    d = temp[temp["LOT_NO"] == lot_no].set_index("MEAS_DT")
    d_1m = d[SENSOR_COLS].resample("1min").mean()  # 1분 평균
    d_1m["LOT_NO"] = lot_no
    pieces.append(d_1m)
temp_1m = pd.concat(pieces).reset_index()
print("1초 자료", len(temp), "행 → 1분 평균", len(temp_1m), "행")

# x축 위치: 이어 붙인 순번, LOT 시작 위치(경계선), LOT 가운데 위치(이름표)
temp_1m["순번"] = range(len(temp_1m))
lot_starts = temp_1m.groupby("LOT_NO")["순번"].min().reindex(lot["LOT_NO"])
lot_centers = temp_1m.groupby("LOT_NO")["순번"].mean().reindex(lot["LOT_NO"])

In [ ]:
# TC(온도) 11채널
fig, axes = plt.subplots(len(TC_COLS), 1, figsize=(15, 1.5 * len(TC_COLS)), sharex=True)
for ax, col in zip(axes, TC_COLS):
    ax.plot(temp_1m["순번"], temp_1m[col], lw=0.7, color="#2a78d6")
    for s in lot_starts:
        ax.axvline(s, color="#9a9a9a", lw=0.6)  # LOT 경계
    ax.set_ylabel(col, rotation=0, ha="right", va="center")
axes[-1].set_xticks(lot_centers.values, xlabels, fontsize=7, rotation=90)  # 짧은 LOT 이름표가 겹치지 않게 세로로
fig.suptitle("TC(온도) 11채널 - LOT를 시간순으로 이어 붙인 1분 평균", y=1.0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "03_overview_tc.png"), bbox_inches="tight")
plt.show()

In [ ]:
# OP(제어출력) 8채널 + CP 2채널 + CLN-IN
other_cols = OP_COLS + CP_COLS + ["CLN-IN"]

fig, axes = plt.subplots(len(other_cols), 1, figsize=(15, 1.5 * len(other_cols)), sharex=True)
for ax, col in zip(axes, other_cols):
    ax.plot(temp_1m["순번"], temp_1m[col], lw=0.7, color="#2a78d6")
    for s in lot_starts:
        ax.axvline(s, color="#9a9a9a", lw=0.6)  # LOT 경계
    ax.set_ylabel(col, rotation=0, ha="right", va="center")
axes[-1].set_xticks(lot_centers.values, xlabels, fontsize=7, rotation=90)
fig.suptitle("OP(제어출력)·CP·CLN-IN - LOT를 시간순으로 이어 붙인 1분 평균", y=1.0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "03_overview_op_cp.png"), bbox_inches="tight")
plt.show()

**읽는 법:** 
- 먼저 **여러 채널(센서값 말하는거임)이 같은 LOT에서 동시에 움직이는지**를 보셈 
    - 여러 채널이 함께 움직이면 조업조건(정지·재가동·강종)의 변화일 가능성이 큼
    - **한 채널만 혼자** 움직이면 그 채널(센서·히터)의 이상 후보로 볼 만함

| 보이는 모양 | 어디서 | 해석 |
| --- | --- | --- |
| Z1~Z8 온도가 **V자로 떨어졌다 회복** | 240058 끝 → 240061 시작, 240114 끝 → 240117 시작 | 여러 채널이 동시에 움직임 → 정기수리 전 **정지**와 수리 후 **재가동**. 이상 후보에서 따로 떼어 봐야 할 구간 |
| OP 전체가 **0으로 떨어짐** | 수리 전 LOT(240058, 240114)의 마지막 약 10분 | 가열을 끈 순간. 수리 후 LOT는 OP가 처음부터 평소 수준 이상으로 시작 |
| TC-Z1·Z2가 **LOT 끝에서 흔들림** | 거의 모든 LOT의 마지막 20~30분 | LOT가 끝날 때마다 반복되는 모양 → 이상이라기보다 LOT 종료 무렵의 조업 변화로 보임 |
| **계단처럼 수준이 바뀜** | TC-Z7·Z8, OP-Z4, CLN-IN의 LOT 경계 | TC-Z7·Z8, OP-Z4는 강종이 바뀔 때 기준 수준이 이동 → 강종별 기준이 필요한 이유. CLN-IN은 데이터 사전상 강종과 무관(같은 강종끼리도 다름) → 이유 확인 필요 |
| **톱니·사인파처럼 주기적으로 움직임** | TC-Z9(톱니), TC-OUT1·OUT2(일정 주기 진동) | 원래 주기적으로 움직이는 채널. 평균만으로는 부족하고 주기·진폭을 봐야 함 |
| **CP-Z4가 크게 튐** | 240058 후반 0.45 → 약 0.1로 하락, 240069에 약 0.9까지 스파이크, 240037에 약 0.6 스파이크 | 이 그림에서 가장 눈에 띄는 이상 후보. 다만 같은 시각 Z4·Z5 존의 OP·TC도 작게 움직임 → **Z4 부근이 함께 반응**한 것일 수 있음 (5-5, 5-6, 6-3에서 확인) |
| 0~100% 사이를 크게 오감 | OP-Z3 | 다른 OP보다 훨씬 불안정. 100% 포화도 잦음 (5-4) |

### 5-4. LOT별 분포 비교 (상자그림)

**왜 이 그래프가 필요한가?**
- **LOT끼리의 수준과 흔들림(산포)** 비교하기 위해서

`BOX_COLS`에 보고 싶은 채널 이름을 넣고 다시 실행하면 됨

In [ ]:
BOX_COLS = ["TC-Z3", "OP-Z3", "TC-Z9", "CP-Z4"]  # ← 보고 싶은 채널로 바꿔서 다시 실행 (2개 이상)

fig, axes = plt.subplots(len(BOX_COLS), 1, figsize=(15, 3 * len(BOX_COLS)), sharex=True)
for ax, col in zip(axes, BOX_COLS):
    data = [temp.loc[temp["LOT_NO"] == n, col].dropna() for n in lot["LOT_NO"]]
    ax.boxplot(data, tick_labels=xlabels, widths=0.6, flierprops=dict(marker=".", markersize=2, alpha=0.3))
    ax.set_ylabel(col)
fig.suptitle("채널별 LOT 분포 (LOT 시간 순서)", y=1.0)
plt.tight_layout()
# plt.savefig(os.path.join(FIG_DIR, "03_box_by_lot.png"), bbox_inches="tight")
plt.show()

**읽는 법:** 
- `TC-Z3`는 수리 전후 4개 LOT(240058·240061·240114·240117)에서 아래쪽 꼬리가 810℃대까지 길게 늘어짐 
- `CP-Z4`는 거의 모든 LOT에서 0.45에 붙어 있는데 
    - **240058만 상자 자체가 0.17 근처까지 내려가** LOT의 상당 부분이 낮은 값
    - 240069는 위로 0.9까지 튄 점이 있다

**왜 OP-Z3 포화 비율을 따로 보는가?**
- OP 100% = 히터가 이미 최대로 일하는 중 → 히터 성능이 떨어지면 포화가 더 잦아짐
- 단, 강종마다 필요한 열량이 다르니 **강종별로** 비교해야 함

In [ ]:
# OP-Z3 출력 포화(100%) 비율 - LOT별
# (temp에 새 컬럼을 만들면 5-2의 OP 채널 목록에 섞이므로, 필요한 두 컬럼만 따로 복사해서 계산)
z3 = temp[["LOT_NO", "OP-Z3"]].copy()
z3["포화"] = z3["OP-Z3"] >= 100
sat = z3.groupby("LOT_NO")["포화"].mean().reindex(lot["LOT_NO"]) * 100

fig, ax = plt.subplots(figsize=(14, 3.5))
bars = ax.bar(xlabels, sat.values, color="#2a78d6")
ax.bar_label(bars, fmt="%.1f", fontsize=8)
ax.set_ylabel("OP-Z3 = 100% 비율 (%)")
ax.set_title("LOT별 OP-Z3 출력 포화 비율")
plt.savefig(os.path.join(FIG_DIR, "03_op_z3_saturation.png"), bbox_inches="tight")
plt.show()

# 강종별로 묶어서 보기
lot["OP-Z3_포화_%"] = sat.values.round(2)
display(lot.groupby("강종")["OP-Z3_포화_%"].agg(["count", "mean", "min", "max"]).round(2))

**읽는 법**
- 포화 비율은 **강종에 따라 크게 갈림**: SPHC 4개 LOT는 모두 0.3% 미만, SPCE·SPCEN·SPFH590은 13~23%
- "포화 많음 = 히터 이상"으로 보면 강종 차이를 이상으로 오인함 → **같은 강종 안에서** 비교 (예: SPCEN 240038 23.1% vs 240114 13.4%)

### 5-5. 이상 후보 지도: 채널 × LOT

**왜 이 그래프가 필요한가?**
- 14 LOT × 21채널을 하나씩 그려 보는 건 비효율적 → 한 장으로 **먼저 들여다볼 칸**을 고르려고

정상 범위 = IQR(Interquartile Range, 사분위 범위) 규칙
- `IQR = Q3 - Q1` (Q1 = 하위 25% 값, Q3 = 상위 25% 값)
- **`Q1 - 3×IQR` ~ `Q3 + 3×IQR`** 밖이면 "범위 밖", 칸 숫자 = 범위 밖 점의 비율(%)
- 강종 보정 안 했고 채널마다 흔들림 크기가 달라서 **같은 행(채널) 안에서 튀는 칸**을 보면 됨 (`CP-Z4M`은 제외)

In [ ]:
CHECK_COLS = [c for c in SENSOR_COLS if c != "CP-Z4M"]

# 채널마다 1년 전체의 사분위수로 정상 범위를 정합니다
q1 = temp[CHECK_COLS].quantile(0.25)
q3 = temp[CHECK_COLS].quantile(0.75)
iqr = q3 - q1
low = q1 - 3 * iqr   # 정상 범위 아래 경계
high = q3 + 3 * iqr  # 정상 범위 위 경계
display(pd.DataFrame({"아래 경계": low, "위 경계": high}).round(3))

# 범위 밖이면 True (low/high는 채널 이름끼리 자동으로 짝지어 비교됩니다)
outside = (temp[CHECK_COLS] < low) | (temp[CHECK_COLS] > high)
outside["LOT_NO"] = temp["LOT_NO"]

# True를 평균 내면 '범위 밖 비율'
ratio = outside.groupby("LOT_NO").mean() * 100  # 행 = LOT, 열 = 채널
ratio = ratio.reindex(lot["LOT_NO"])             # LOT 시간 순서로
ratio = ratio.T                                  # 행 = 채널, 열 = LOT 로 뒤집기

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
im = ax.imshow(ratio.values, cmap="Blues", aspect="auto")

# 5% 이상인 칸에만 숫자 표시
for i in range(len(CHECK_COLS)):
    for j in range(len(lot)):
        v = ratio.iloc[i, j]
        if v >= 5:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                    color="white" if v >= 25 else "black")

ax.set_xticks(range(len(lot)), xlabels, fontsize=8)
ax.set_yticks(range(len(CHECK_COLS)), CHECK_COLS)
ax.grid(False)
fig.colorbar(im, label="정상 범위 밖 비율 (%)")
ax.set_title("이상 후보 지도: 1년 기준 정상 범위(Q1-3×IQR ~ Q3+3×IQR) 밖 비율")
# plt.savefig(os.path.join(FIG_DIR, "03_outlier_map.png"), bbox_inches="tight")
plt.show()

display(ratio.round(1))

**읽는 법**
- `TC-Z4~Z6`: **정기수리 전후 4개 LOT(240058·240061·240114·240117)에서만** 15~38%, 나머지 0% → 정지·재가동 구간이 잡힌 것
- `CP-Z4`: **240058이 약 44%로 가장 진함**, 나머지는 한 자릿수(240069 약 9% 등)
    - 정상 범위 폭이 약 0.018로 아주 좁아서 작은 흔들림도 잡힘 → 한 자릿수 칸은 크게 믿지 않음
- `TC-Z1~Z2`가 짧은 LOT(240038·240044·240133)에서 진한 건, 모든 LOT 끝 20~30분에 흔들림이 있어서 LOT가 짧을수록 비율이 커지기 때문
- `TC-Z9`, `TC-OUT1·OUT2`의 0%는 원래 크게 움직이는 채널이라 범위가 넓어서임. "이상 없음"이 아님

### 5-6. LOT 하나 자세히 보기

**왜 이 그래프가 필요한가?**
- 지도는 "어디를 볼지"만 알려줌 → 거기서 **무슨 일이 어떤 순서로** 일어났는지는 1초 원자료로 봐야 함

| 표시 | 의미 | 잘 잡는 것 |
| --- | --- | --- |
| 회색 음영 | 5-5의 1년 기준 정상 범위 | 정지·재가동처럼 **크고 오래 가는** 이탈 |
| 빨간 점 | 10분 이동평균에서 3σ(표준편차 3배) 넘게 벗어난 점 | **몇 초만** 튄 값 (몇 분 가는 스파이크는 놓침) |

- 둘 다 **"이상 후보"일 뿐** 고장 판정 아님
- 예시는 지도에서 `CP-Z4`가 가장 진했던 **LOT 240058 (6월 노내 정비 전날)**. `LOT_PICK` 바꿔서 다른 LOT도 볼 수 있음

In [ ]:
LOT_PICK = 240058                                  # ← 보고 싶은 LOT 번호 (예: 240061)
PICK_COLS = ["TC-Z3", "TC-Z6", "OP-Z3", "CP-Z4"]   # ← 보고 싶은 채널 (2개 이상, CP-Z4M 제외)

d = temp[temp["LOT_NO"] == LOT_PICK].set_index("MEAS_DT")

fig, axes = plt.subplots(len(PICK_COLS), 1, figsize=(14, 2.3 * len(PICK_COLS)), sharex=True)
for ax, col in zip(axes, PICK_COLS):
    s = d[col]
    mean = s.rolling("10min", center=True, min_periods=60).mean()  # 10분 이동평균
    std = s.rolling("10min", center=True, min_periods=60).std()    # 10분 이동표준편차
    out = (s - mean).abs() > 3 * std                                # 3σ 밖이면 True

    ax.plot(s.index, s, lw=0.5, color="#b7d3f6", label="1초 원자료")
    ax.plot(mean.index, mean, lw=1.3, color="#2a78d6", label="10분 이동평균")
    ax.scatter(s.index[out], s[out], s=10, color="#d03b3b", zorder=3, label=f"3σ 밖 ({out.sum()}점)")

    ylim = ax.get_ylim()  # 자료가 있는 y 범위를 기억해 두고
    ax.axhspan(low[col], high[col], color="#e1e0d9", alpha=0.6, zorder=0, label="1년 기준 정상 범위")
    ax.set_ylim(ylim)     # 음영 때문에 y축이 늘어나지 않게 되돌림
    ax.set_ylabel(col)
    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)

axes[0].set_title(f"LOT {LOT_PICK} ({grade_of[LOT_PICK]}) - 1초 원자료와 두 가지 기준선")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
plt.tight_layout()
# plt.savefig(os.path.join(FIG_DIR, f"03_lot_{LOT_PICK}.png"), bbox_inches="tight")
plt.show()

**읽는 법 (LOT 240058, 6월 9일)**
- 07:33 `CP-Z4`가 먼저 0.45 → 0.4 아래로 떨어지기 시작 (LOT 끝엔 약 0.1)
    - 같은 시각 `OP-Z4~Z8`, `TC-Z4`도 평소 범위를 벗어남
- `TC-Z3·Z6`는 08:05 무렵부터 떨어짐 → 09:02 OP 8개 모두 0 → 다음 날(6/10)부터 노내 정비

### `CP-Z4`·`OP-Z4~Z8`이 온도보다 약 30~50분 먼저 움직였음
- 단, 노내 정비 전 정지 절차(분위기 가스 교체)일 수도 있음 (가정, 확인 필요) → 고장 징후로 단정 못 함
- 11월 수리 전 LOT(240114)에선 `CP-Z4`가 안 떨어짐 → 노내 정비 vs 라인 전체 수리라는 `SCOPE` 차이와 맞음
- 빨간 점(3σ)은 이 큰 변화들에 **하나도 안 찍힘** → 천천히 크게 변하면 이동평균도 같이 따라가서 못 잡음

## 6. 온도 × 정비 × 부품을 한 장으로 보기

### 6-1. 2024년 타임라인

**왜 이 그래프가 필요한가?**
- 따로 본 정비·부품·온도를 **같은 날짜 축에** 올려서, 서로 얼마나 가까운지 한눈에 보려고

그래프 보는 법: 맨 위 = 온도 LOT(+강종), 가운데 = 정비 탐지(TBM·BM·CBM), 맨 아래 = 부품 교체(별), 회색 = 정기수리

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))

for _, r in cr1_repair.iterrows():
    ax.axvspan(r["STRT_DT"], r["END_DT"] + pd.Timedelta(days=1), color="#d9d9d9", lw=0)

# 온도 LOT (y = 4)
for _, r in lot.iterrows():
    ax.plot([r["시작"], r["시작"]], [3.6, 4.4], color="#52514e", lw=2)
    ax.text(r["시작"], 4.5, r["강종"], rotation=90, fontsize=7, ha="center", va="bottom")

# 정비 탐지 (y = 1, 2, 3)
for i, t in enumerate(MNT_ORDER, start=1):
    sub = cal[cal["MNT_TYPE"] == t]
    ax.scatter(sub["DETC_DT"], [i] * len(sub), color=MNT_COLORS[t], s=45, edgecolor="white", zorder=3, label=t)

# 부품 교체 (y = 0)
ax.scatter(cal_part["END_DT"], [0] * len(cal_part), marker="*", s=160, color="#52514e", zorder=3, label="부품교체")

ax.set_yticks([0, 1, 2, 3, 4], ["부품교체", "TBM", "BM", "CBM", "온도 LOT"])
ax.set_ylim(-0.7, 5.6)
ax.set_xlim(pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
ax.set_title("연속소둔로 2024: 온도 관측 LOT · 정비 · 부품교체 (회색 = CR1 정기수리)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
# plt.savefig(os.path.join(FIG_DIR, "03_timeline_all.png"), bbox_inches="tight")
plt.show()

**읽는 법**
- TBM은 정기수리 구간에 몰려 있고, 부품 교체는 3~8월·12월 말에 흩어져 있음
- 온도 LOT는 1년에 14번, 짧게만 관측돼서 정비 날짜와 거의 안 겹침 → 6-2에서 숫자로 확인

### 6-2. 정비는 온도 관측 직후에 일어났나?

**왜 이 계산과 그래프가 필요한가?**
- 정비가 온도 관측 구간과 **시간상 가까워야** 그 LOT의 온도로 정비를 설명할 수 있음
- 정비마다 탐지 직전 LOT를 찾아서 LOT 끝 → 탐지까지 몇 시간인지 계산 (정기수리 계획 작업은 빼고도 계산)

In [ ]:
rows = []
for _, m in cal.iterrows():
    before = lot[lot["시작"] <= m["DETC_DT"]]  # 탐지 시각 전에 시작한 LOT들
    last = before.iloc[-1]                     # 그중 가장 최근 LOT
    rows.append({"WO_NO": m["WO_NO"], "LOT_NO": last["LOT_NO"], "강종": last["강종"], "LOT끝": last["끝"]})

near = cal.merge(pd.DataFrame(rows), on="WO_NO")
near["LOT중_탐지"] = near["DETC_DT"] <= near["LOT끝"]
near["LOT끝→탐지_h"] = (near["DETC_DT"] - near["LOT끝"]).dt.total_seconds() / 3600
near.loc[near["LOT중_탐지"], "LOT끝→탐지_h"] = 0  # LOT 관측 중에 탐지된 경우는 0

after = near[~near["LOT중_탐지"]]  # LOT가 끝난 뒤 탐지된 정비
print("정비", len(near), "건 중")
print(" - LOT 관측 중에 탐지:", near["LOT중_탐지"].sum(), "건")
print(" - LOT 끝난 뒤 1시간 이내:", (after["LOT끝→탐지_h"] <= 1).sum(), "건")
print(" - LOT 끝난 뒤 24시간 이내:", (after["LOT끝→탐지_h"] <= 24).sum(), "건")
print(f" - LOT 끝 → 탐지 시간 중앙값: {after['LOT끝→탐지_h'].median():.1f}시간 ({after['LOT끝→탐지_h'].median() / 24:.1f}일)")
print(" - 뒤에 정비가 따라온 LOT 수:", near["LOT_NO"].nunique(), "/", len(lot))

# 정기수리 기간의 계획 작업을 빼고 다시 계산
after2 = after[~after["정기수리중"]]
print("정기수리 기간 정비를 빼면:", len(after2), "건 중")
print(" - LOT 끝난 뒤 24시간 이내:", (after2["LOT끝→탐지_h"] <= 24).sum(), "건")
print(f" - LOT 끝 → 탐지 시간 중앙값: {after2['LOT끝→탐지_h'].median():.1f}시간 ({after2['LOT끝→탐지_h'].median() / 24:.1f}일)")
print(" - 뒤에 정비가 따라온 LOT 수:", near.loc[~near["정기수리중"], "LOT_NO"].nunique(), "/", len(lot))

display(near.sort_values("LOT끝→탐지_h").head(8)[
    ["WO_NO", "DETC_DT", "MNT_TYPE", "FAIL_MODE", "PART_CHG", "REMARK", "정기수리중", "LOT_NO", "강종", "LOT끝", "LOT끝→탐지_h"]].round({"LOT끝→탐지_h": 1}))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
for i, t in enumerate(MNT_ORDER):
    sub = after[after["MNT_TYPE"] == t]
    ax.scatter(sub["LOT끝→탐지_h"] / 24, [i] * len(sub), color=MNT_COLORS[t], s=45, edgecolor="white", label=t)
ax.set_yticks([0, 1, 2], MNT_ORDER)
ax.set_ylim(-0.7, 2.7)
ax.set_xlabel("직전 LOT가 끝나고 정비를 탐지하기까지 (일)")
ax.set_title("정비 탐지는 온도 관측이 끝나고 얼마 뒤인가 (LOT 관측 중 탐지 건은 제외)")
# plt.savefig(os.path.join(FIG_DIR, "03_lot_end_to_detect.png"), bbox_inches="tight")
plt.show()

**읽는 법**
- LOT 관측 중 탐지는 2건뿐, 나머지 42건은 LOT 끝나고 **중앙값 약 4.9일 뒤** (최소 약 2.3시간, 1시간 이내 0건)
- 정기수리 계획 작업 11건을 빼면 중앙값 **약 9일**, 24시간 이내 5건

### 정비를 특정 LOT에 1:1로 붙이긴 어려움 → 가장 가까운 사례부터 보자

### 6-3. 가장 가까운 사례 자세히 보기

**왜 이 그래프가 필요한가?**
- "정비 직전 온도·출력이 실제로 달랐나?"를 확인할 수 있는 **몇 안 되는 사례**
- 5-6과 같은 방법으로 그리고, 정비 탐지 시각에 주황색 세로선

첫 번째 사례: **LOT 240133 관측 중 CBM(WER, 마모량 초과) 탐지** → 가스 분석계 셀 예방 교체

In [ ]:
LOT_PICK = 240133
PICK_COLS = ["TC-Z3", "TC-Z6", "OP-Z3", "CP-Z4"]
wo = near[near["WO_NO"] == "WO-2024-02105"].iloc[0]  # 이 LOT 관측 중에 탐지된 정비
print(wo["WO_NO"], wo["DETC_DT"], wo["MNT_TYPE"], wo["FAIL_MODE"], wo["REMARK"])

d = temp[temp["LOT_NO"] == LOT_PICK].set_index("MEAS_DT")

fig, axes = plt.subplots(len(PICK_COLS), 1, figsize=(14, 2.3 * len(PICK_COLS)), sharex=True)
for ax, col in zip(axes, PICK_COLS):
    s = d[col]
    mean = s.rolling("10min", center=True, min_periods=60).mean()
    std = s.rolling("10min", center=True, min_periods=60).std()
    out = (s - mean).abs() > 3 * std

    ax.plot(s.index, s, lw=0.5, color="#b7d3f6", label="1초 원자료")
    ax.plot(mean.index, mean, lw=1.3, color="#2a78d6", label="10분 이동평균")
    ax.scatter(s.index[out], s[out], s=10, color="#d03b3b", zorder=3, label=f"3σ 밖 ({out.sum()}점)")
    ax.axvline(wo["DETC_DT"], color="#eb6834", lw=2, label="정비 탐지 DETC_DT")

    ylim = ax.get_ylim()
    ax.axhspan(low[col], high[col], color="#e1e0d9", alpha=0.6, zorder=0, label="1년 기준 정상 범위")
    ax.set_ylim(ylim)
    ax.set_ylabel(col)
    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)

axes[0].set_title(f"LOT {LOT_PICK} ({grade_of[LOT_PICK]}) - 주황선 = 정비 탐지 시각")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
plt.tight_layout()
# plt.savefig(os.path.join(FIG_DIR, f"03_lot_{LOT_PICK}_with_mnt.png"), bbox_inches="tight")
plt.show()

**읽는 법 (LOT 240133)**
- 탐지 40~50분 전(15:06~15:12) `TC-Z3`가 약 872℃로 오르자 `OP-Z3`가 약 13%로 떨어지고, 850℃로 내려가자 100%로 붙음 → **온도 넘치면 출력 줄이고, 모자라면 올리는 제어 반응**
- LOT 시작(14:57, 강종변경) 직후라 강종 전환 과정일 수도 있음
- 탐지 시각엔 모두 안정적 → 정비 내용과 연결할 근거 없음

두 번째 사례: **LOT 240069 종료 약 2.3시간 뒤 BM(ELE, 절연 저하) 탐지** → O링 세트 예방 교체 (탐지가 LOT 밖이라 세로선 없음)

In [ ]:
LOT_PICK = 240069
PICK_COLS = ["TC-Z3", "TC-Z6", "OP-Z3", "CP-Z4"]
wo = near[near["WO_NO"] == "WO-2024-01094"].iloc[0]
print(wo["WO_NO"], wo["DETC_DT"], wo["MNT_TYPE"], wo["FAIL_MODE"], wo["REMARK"], f"LOT 종료 {wo['LOT끝→탐지_h']:.1f}시간 뒤")

d = temp[temp["LOT_NO"] == LOT_PICK].set_index("MEAS_DT")

fig, axes = plt.subplots(len(PICK_COLS), 1, figsize=(14, 2.3 * len(PICK_COLS)), sharex=True)
for ax, col in zip(axes, PICK_COLS):
    s = d[col]
    mean = s.rolling("10min", center=True, min_periods=60).mean()
    std = s.rolling("10min", center=True, min_periods=60).std()
    out = (s - mean).abs() > 3 * std

    ax.plot(s.index, s, lw=0.5, color="#b7d3f6", label="1초 원자료")
    ax.plot(mean.index, mean, lw=1.3, color="#2a78d6", label="10분 이동평균")
    ax.scatter(s.index[out], s[out], s=10, color="#d03b3b", zorder=3, label=f"3σ 밖 ({out.sum()}점)")

    ylim = ax.get_ylim()
    ax.axhspan(low[col], high[col], color="#e1e0d9", alpha=0.6, zorder=0, label="1년 기준 정상 범위")
    ax.set_ylim(ylim)
    ax.set_ylabel(col)
    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)

axes[0].set_title(f"LOT {LOT_PICK} ({grade_of[LOT_PICK]}) - 이 LOT 종료 약 2.3시간 뒤 BM 탐지")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
plt.tight_layout()
# plt.savefig(os.path.join(FIG_DIR, f"03_lot_{LOT_PICK}.png"), bbox_inches="tight")
plt.show()

**읽는 법 (LOT 240069)**
- 7/8 02:31 **`CP-Z4`가 약 0.91까지 튐** (BM 탐지 약 6.3시간 전)
    - 같은 시각 `OP-Z4`, `TC-Z4·Z5`도 LOT 최솟값·최댓값을 찍음 → CP-Z4 혼자가 아니라 **Z4·Z5 존이 같이 반응**한 이벤트 후보
    - 확인하려면 `PICK_COLS = ["CP-Z4", "OP-Z4", "TC-Z4", "TC-Z5"]`
- `TC-Z3` 큰 흔들림(23:40 약 850℃, 04:54 약 871℃)은 240133과 같은 제어 반응 모양
- 고장은 전기 계통(절연 저하), 교체 부품은 O링이라 직접 연결 근거는 없음 → "CP-Z4 스파이크 뒤 BM"이 반복되는지 볼 **후보**로만 기록

### 6-4. LOT 대표값의 변화 (정기수리 전후)

**왜 이 그래프가 필요한가?**
- 6월 노내 정비 후 기준 수준이 바뀌었다면, 1년치를 정상 범위 하나로 묶은 5-5가 틀어짐
- LOT별 중앙값(점)과 25~75% 범위(막대)로 **같은 강종끼리** 수리 전후를 비교

In [ ]:
REP_COLS = ["TC-Z3", "TC-Z6", "TC-Z7", "TC-Z9", "TC-OUT1", "OP-Z3"]

lot_med = temp.groupby("LOT_NO")[REP_COLS].median().reindex(lot["LOT_NO"])        # 중앙값
lot_q1 = temp.groupby("LOT_NO")[REP_COLS].quantile(0.25).reindex(lot["LOT_NO"])  # 25%
lot_q3 = temp.groupby("LOT_NO")[REP_COLS].quantile(0.75).reindex(lot["LOT_NO"])  # 75%
x = lot["시작"].values  # LOT 시작 날짜

fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharex=True)
for ax, col in zip(axes.flat, REP_COLS):
    for _, r in cr1_repair.iterrows():
        ax.axvspan(r["STRT_DT"], r["END_DT"] + pd.Timedelta(days=1), color="#d9d9d9", lw=0)
    ax.vlines(x, lot_q1[col], lot_q3[col], color="#9ec5f4", lw=2)  # 25~75% 범위 (세로 막대)
    ax.scatter(x, lot_med[col], color="#2a78d6", zorder=3)          # 중앙값 (점)
    for xi, yi, g in zip(x, lot_med[col], lot["강종"]):
        ax.annotate(g, (xi, yi), xytext=(4, 4), textcoords="offset points", fontsize=7)
    ax.set_title(col)

for ax in axes[-1]:
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
fig.suptitle("LOT 중앙값(점)과 25~75% 범위(막대), 회색 = CR1 정기수리", y=1.0)
plt.tight_layout()
# plt.savefig(os.path.join(FIG_DIR, "03_lot_level_trend.png"), bbox_inches="tight")
plt.show()

# 숫자로도: 강종 순으로 정렬한 LOT 중앙값
table = lot_med.round(1)
table["강종"] = grade_of
display(table.sort_values(["강종", "LOT_NO"]))

**읽는 법**
- **수리 시점과 맞물려 수준이 바뀐 채널은 안 보임** (예: SPCE `OP-Z3` 87.7 → 87.7 → 87.6)
    - 단, 강종마다 수리 전 LOT가 1개뿐이라 근거는 약함
- `TC-Z7`은 같은 강종 안에서도 약 3℃씩 다름 (예: SPCE 330.2 → 333.3 → 330.1) → **강종만으로 설명 안 되는 설정 차이**
- `TC-Z9`·`TC-OUT1`처럼 주기적으로 움직이는 채널은 LOT가 짧으면 중앙값이 들쭉날쭉해서 비교 주의

## 7. 정리

### 순서대로 따라간 결과

| 단계 | 결과 |
| --- | --- |
| 정비이력 → CAL01 | 2,112건 중 44건 (TBM 22 · BM 12 · CBM 10), 부품교체 Y 8건 |
| → `WO_NO` → 부품교체이력 | 정비 8건 → 교체 9행 (정비 1 : 부품 N), 두 파일의 `PART_CHG`와 정확히 일치 |
| → `PART_CD` → 부품마스터 | 실제 교체 부품 3종: 오일씰 SEL-006, O링 세트 SEL-009, 가스 분석계 셀 INS-006 (각 3회) |

### 알게 된 것

**정비·부품**
- 탐지→착수 대기시간은 **`MNT_TYPE`이 가름** (TBM·BM 수 시간, CBM 중앙값 약 5일). CBM의 긴 대기가 "계획된 여유"인지는 불확실
- `FAIL_MODE` 공란은 결측 아님 (24건 중 20건이 TBM)
- 교체 부품 3종 모두 표준수명 40% 미만 간격으로 재교체 → 단 손상 교체는 6번 중 3번이라 "조기 교체 후보"까지만
- 히터·열전대 등 온도 직결 부품은 교체 기록 없음 → 부품 교체로 온도 이상을 검증하긴 어려움

**온도**
- 여러 채널이 동시에 크게 움직이는 곳 = **정기수리 전 정지·후 재가동** (240058·240061·240114·240117) → 따로 떼어 봐야 함
- **`CP-Z4`가 가장 눈에 띄는 이상 후보** (240058 하락, 240069 스파이크). 단 Z4·Z5 존 채널도 같이 움직임
- `OP-Z3` 포화 비율은 이 데이터에선 **강종에 따라 크게 갈림** → 강종 안에서 비교
- 3σ는 몇 초짜리 튐만, IQR 범위는 큰 이탈만 잡음 → 기준선마다 놓치는 게 다름
- 정비 탐지는 대부분 LOT 끝나고 며칠 뒤 → 정비를 LOT에 1:1로 붙이긴 무리
- `CP-Z4M` = `TC-Z5` 중복 컬럼 → 데이터 사전에도 명시돼 있어 제외 근거 확인됨
- 확인 필요: 정기수리 중 강종변경 이벤트 4건, `DETC_DT`가 TBM에도 있음

### 다음 단계
- LOT를 정지·재가동 / 강종 전환 직후 / LOT 종료 직전 / 안정 구간으로 나누기
- 같은 강종 안에서 비교하는 기준선 만들기
- 특징 후보: 10분 이동평균·표준편차, 변화율, `OP-Z3` 포화 비율, `CP-Z4` 스파이크 횟수, 같은 존 TC–OP 관계, `TC-Z9`·`TC-OUT` 주기·진폭
- 두 기준선(3σ, IQR)과 IsolationForest 비교 → 6-3 사례부터 확인

### 한계
- `QTY`는 데이터 사전상 "한 작업지시에서 해당 부품을 교체한 수량(개)" → 가정 아님
- 확인된 정의가 없어서 **가정 또는 확인 필요**로 남는 것
    - CP 채널의 구체적 의미와 단위 (사전: 카본포텐셜, 단위 `[확인필요]`)
    - 표준수명(`STD_LIFE_H`)이 가동시간 기준인지 달력시간 기준인지
    - 240058의 CP 하락이 정지 절차인지
    - 강종변경 이벤트의 구체적인 업무 의미 (투입 시각인지 계획 시각인지)
- 고장모드 배정, 부품 수명·리드타임, 정기수리 일정, 이벤트 내용은 데이터 사전상 설정값(`[가정]`)
- 온도는 14개 LOT·약 99시간뿐, 강종당 LOT 1~4개 → 강종 효과와 LOT 효과를 완전히 분리할 수 없고, **결과를 일반화하지 않음**
- 본 데이터는 실제 조업 데이터가 아니라 KAMP 공개 제조AI데이터셋을 재구성한 **교육용 재현 데이터**